# Week 8 Intern Mini Project - E-Commerce Order Analytics System
Skills Tested: Python, SQL, Problem Solving

Phase 1: Build everything using Python and SQL (local environment)

In [32]:
import pandas as pd
import numpy as np
import random
import sqlite3
import re
from datetime import datetime, timedelta
from faker import Faker

fake = Faker()
random.seed(42)
np.random.seed(42)

import os
os.makedirs("data", exist_ok=True)
os.makedirs("output", exist_ok=True)

# Part 1: Data Generation

**customers.csv**

In [33]:
num_customers = 520
customer_types = ["REGULAR", "PREMIUM", "VIP"]

customers = []
for i in range(1, num_customers + 1):
    email = fake.email()

    # about 2% invalid emails - just strip the @ so it fails validation
    if random.random() < 0.02:
        email = email.replace("@", "")

    customers.append({
        "customer_id": i,
        "customer_name": fake.name(),
        "email": email,
        "registration_date": fake.date_between(start_date="-2y", end_date="-30d"),
        "customer_type": random.choice(customer_types)
    })

customers_df = pd.DataFrame(customers)
customers_df.to_csv("data/customers.csv", index=False)
customers_df.head()

,customer_id,customer_name,email,registration_date,customer_type
0,1,Shannon Pacheco,karenstafford@example.org,2024-12-01,REGULAR
1,2,Jose Becker,adam63@example.net,2025-08-30,REGULAR
2,3,Rebecca Barr,wellsnicole@example.com,2025-09-09,VIP
3,4,Lori Diaz,gbennett@example.com,2026-04-19,VIP
4,5,David Stevens,sharon23@example.net,2026-07-09,REGULAR


**products.csv**

In [34]:
categories = {
    "Electronics": ["Mobiles", "Laptops", "Accessories"],
    "Clothing": ["Men", "Women", "Kids"],
    "Home": ["Furniture", "Kitchen", "Decor"],
    "Books": ["Fiction", "Non-Fiction", "Comics"]
}

products = []
product_id = 1
for category, subcats in categories.items():
    for subcat in subcats:
        for _ in range(45):
            name = fake.word().title() + " " + subcat[:-1] if subcat.endswith("s") else fake.word().title()
            products.append({
                "product_id": product_id,
                "product_name": name,
                "category": category,
                "subcategory": subcat,
                "cost_price": round(random.uniform(5, 500), 2)
            })
            product_id += 1

products_df = pd.DataFrame(products)

# mess up ~10% of names on purpose so clean_products() has something to fix
messy_idx = products_df.sample(frac=0.1, random_state=1).index
products_df.loc[messy_idx, "product_name"] = products_df.loc[messy_idx, "product_name"].apply(
    lambda x: "  " + x.upper() + "  "
)

products_df.to_csv("data/products.csv", index=False)
products_df.head()

,product_id,product_name,category,subcategory,cost_price
0,1,Popular Mobile,Electronics,Mobiles,342.49
1,2,Skill Mobile,Electronics,Mobiles,94.13
2,3,Activity Mobile,Electronics,Mobiles,91.66
3,4,Behavior Mobile,Electronics,Mobiles,306.70
4,5,Call Mobile,Electronics,Mobiles,194.36


**orders.csv**

In [35]:
num_orders = 600
statuses = ["PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"]
region_codes = ["N", "S", "E", "W", "C"]

orders = []
for order_id in range(1, num_orders + 1):
    customer_id = random.choice(customers_df["customer_id"].tolist())

    # roughly 5% should have no customer_id
    if random.random() < 0.05:
        customer_id = None

    order_date = fake.date_time_between(start_date="-1y", end_date="now")

    # a chunk of these are in DD-MM-YYYY instead of the normal format - on purpose
    if random.random() < 0.1:
        order_date_str = order_date.strftime("%d-%m-%Y %H:%M:%S")
    else:
        order_date_str = order_date.strftime("%Y-%m-%d %H:%M:%S")

    orders.append({
        "order_id": order_id,
        "customer_id": customer_id,
        "order_date": order_date_str,
        "status": random.choice(statuses),
        "region_code": random.choice(region_codes)
    })

orders_df = pd.DataFrame(orders)
orders_df.to_csv("data/orders.csv", index=False)
orders_df.head()

,order_id,customer_id,order_date,status,region_code
0,1,148.0,20-10-2025 15:37:22,DELIVERED,S
1,2,427.0,2026-05-10 13:58:43,RETURNED,E
2,3,70.0,2026-02-21 10:47:44,RETURNED,C
3,4,21.0,2026-07-25 14:42:03,CANCELLED,E
4,5,260.0,2025-10-01 20:49:08,PLACED,E


**order_items.csv**

In [36]:
order_ids = orders_df["order_id"].tolist()
product_ids = products_df["product_id"].tolist()

num_items = 1500
order_items = []
for item_id in range(1, num_items + 1):
    quantity = random.randint(1, 5)

    # ~3% are returns, so quantity goes negative
    if random.random() < 0.03:
        quantity = -quantity

    order_items.append({
        "item_id": item_id,
        "order_id": random.choice(order_ids),   # has to be a real order_id
        "product_id": random.choice(product_ids),
        "quantity": quantity,
        "unit_price": round(random.uniform(10, 1000), 2),
        "discount_percent": round(random.uniform(0, 100), 2)
    })

order_items_df = pd.DataFrame(order_items)
order_items_df.to_csv("data/order_items.csv", index=False)
order_items_df.head()

,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,1,256,365,2,164.44,10.67
1,2,505,7,1,766.60,54.15
2,3,381,323,1,497.42,98.54
3,4,503,188,5,984.55,91.60
4,5,591,132,3,63.61,79.93


# Part 2: Data Cleaning

**clean_orders()** - fixes the two date formats and flags rows with missing customer_id

In [37]:
def clean_orders(df):
    df = df.copy()

    def parse_date(x):
        for fmt in ("%Y-%m-%d %H:%M:%S", "%d-%m-%Y %H:%M:%S"):
            try:
                return datetime.strptime(x, fmt)
            except ValueError:
                continue
        return pd.NaT

    df["order_date"] = df["order_date"].apply(parse_date)
    df["customer_id"] = df["customer_id"].replace("", np.nan)
    df["missing_customer"] = df["customer_id"].isna()

    return df

orders_clean = clean_orders(orders_df)
orders_clean.head()

,order_id,customer_id,order_date,status,region_code,missing_customer
0,1,148.0,2025-10-20 15:37:22,DELIVERED,S,False
1,2,427.0,2026-05-10 13:58:43,RETURNED,E,False
2,3,70.0,2026-02-21 10:47:44,RETURNED,C,False
3,4,21.0,2026-07-25 14:42:03,CANCELLED,E,False
4,5,260.0,2025-10-01 20:49:08,PLACED,E,False


**clean_products()** - strips extra spaces and title-cases the names

In [38]:
def clean_products(df):
    df = df.copy()
    df["product_name"] = df["product_name"].str.strip().str.title()
    return df

products_clean = clean_products(products_df)
products_clean.head()

,product_id,product_name,category,subcategory,cost_price
0,1,Popular Mobile,Electronics,Mobiles,342.49
1,2,Skill Mobile,Electronics,Mobiles,94.13
2,3,Activity Mobile,Electronics,Mobiles,91.66
3,4,Behavior Mobile,Electronics,Mobiles,306.70
4,5,Call Mobile,Electronics,Mobiles,194.36


**validate_emails()** - checks email format, returns customer_ids that fail

In [39]:
def validate_emails(df):
    pattern = r"^[\w\.-]+@[\w\.-]+\.\w+$"
    invalid = df[~df["email"].str.match(pattern)]
    return invalid["customer_id"].tolist()

invalid_email_ids = validate_emails(customers_df)
print("Invalid email customer_ids:", invalid_email_ids)

Invalid email customer_ids: [68, 152, 174, 463, 498]


**check_referential_integrity()** - order_items rows whose order_id doesn't exist in orders

In [40]:
def check_referential_integrity(orders_df, order_items_df):
    valid_ids = set(orders_df["order_id"])
    orphan_items = order_items_df[~order_items_df["order_id"].isin(valid_ids)]
    return orphan_items

orphans = check_referential_integrity(orders_df, order_items_df)
print("Orphan order_items rows:", len(orphans))

Orphan order_items rows: 0


Save the cleaned files and a small report

In [41]:
orders_clean.to_csv("output/orders_clean.csv", index=False)
products_clean.to_csv("output/products_clean.csv", index=False)

report = f"""
Data Cleaning Report
---------------------
Orders with missing customer_id : {orders_clean['missing_customer'].sum()}
Invalid emails                  : {len(invalid_email_ids)}
Negative quantity rows          : {(order_items_df['quantity'] < 0).sum()}
Orphan order_items rows         : {len(orphans)}
"""

print(report)

with open("output/cleaning_report.txt", "w") as f:
    f.write(report)


Data Cleaning Report
---------------------
Orders with missing customer_id : 30
Invalid emails                  : 5
Negative quantity rows          : 58
Orphan order_items rows         : 0



# Part 3: SQL Analysis (SQLite)

In [42]:
conn = sqlite3.connect("ecommerce.db")

orders_clean.to_sql("orders", conn, if_exists="replace", index=False)
products_clean.to_sql("products", conn, if_exists="replace", index=False)
customers_df.to_sql("customers", conn, if_exists="replace", index=False)
order_items_df.to_sql("order_items", conn, if_exists="replace", index=False)

print("Tables loaded into SQLite")

Tables loaded into SQLite


**Basic Queries**

**Q1** - Total revenue per category

In [43]:
query1 = """
SELECT p.category,
       SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
"""

pd.read_sql(query1, conn)

,category,total_revenue
0,Books,278809.087436
1,Clothing,249549.007414
2,Electronics,247145.014270
3,Home,234769.515829


**Q2** - Top 10 customers by total order value

In [44]:
query2 = """
SELECT o.customer_id,
       SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS total_value
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.customer_id IS NOT NULL
GROUP BY o.customer_id
ORDER BY total_value DESC
LIMIT 10;
"""

pd.read_sql(query2, conn)

,customer_id,total_value
0,430.0,14298.559379
1,257.0,11749.655188
2,406.0,11629.061731
3,61.0,11227.834712
4,356.0,11108.293241
5,322.0,9765.622630
6,501.0,9750.286897
7,173.0,9663.408658
8,138.0,9639.078772
9,487.0,9631.854630


**Q3** - Month-wise order count for the last 12 months

In [45]:
query3 = """
SELECT strftime('%Y-%m', order_date) AS month,
       COUNT(*) AS order_count
FROM orders
WHERE order_date >= date('now', '-12 months')
GROUP BY month
ORDER BY month;
"""

pd.read_sql(query3, conn)

,month,order_count
0,2025-08,38
1,2025-09,37
2,2025-10,70
3,2025-11,52
4,2025-12,43
5,2026-01,50
6,2026-02,51
7,2026-03,46
8,2026-04,43
9,2026-05,55


**Intermediate Queries**

**Q4** - Customers who placed orders but never had any item delivered

In [46]:
query4 = """
SELECT DISTINCT customer_id
FROM orders
WHERE customer_id IS NOT NULL
AND customer_id NOT IN (
    SELECT customer_id FROM orders WHERE status = 'DELIVERED'
);
"""

pd.read_sql(query4, conn)

,customer_id


**Q5** - Products that had more returns than purchases

In [47]:
query5 = """
SELECT product_id,
       SUM(CASE WHEN quantity > 0 THEN quantity ELSE 0 END) AS purchased,
       SUM(CASE WHEN quantity < 0 THEN -quantity ELSE 0 END) AS returned
FROM order_items
GROUP BY product_id
HAVING returned > purchased;
"""

pd.read_sql(query5, conn)

,product_id,purchased,returned
0,60,0,2
1,207,4,5
2,232,3,4
3,321,1,5
4,390,2,5
5,395,1,5
6,428,3,4
7,431,0,4


**Q6** - Return rate per category

In [48]:
query6 = """
SELECT p.category,
       SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) * 1.0 /
       SUM(ABS(oi.quantity)) AS return_rate
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category;
"""

pd.read_sql(query6, conn)

,category,return_rate
0,Books,0.026957
1,Clothing,0.050000
2,Electronics,0.030584
3,Home,0.051188


**Advanced Queries** (window functions, CTEs, subqueries)

**Q7** - Running total of revenue per region, ordered by date

In [49]:
query7 = """
WITH daily AS (
    SELECT o.region_code,
           date(o.order_date) AS order_date,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS daily_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.region_code, date(o.order_date)
)
SELECT region_code, order_date, daily_revenue,
       SUM(daily_revenue) OVER (PARTITION BY region_code ORDER BY order_date) AS running_total
FROM daily
ORDER BY region_code, order_date;
"""

pd.read_sql(query7, conn)

,region_code,order_date,daily_revenue,running_total
0,C,2025-08-09,2103.589711,2103.589711
1,C,2025-08-18,3218.001684,5321.591395
2,C,2025-08-20,387.669888,5709.261283
3,C,2025-08-30,2617.923429,8327.184712
4,C,2025-09-02,1576.699982,9903.884694
...,...,...,...,...
466,W,2026-07-21,3337.902516,197673.202827
467,W,2026-07-27,8610.056411,206283.259238
468,W,2026-07-31,924.669830,207207.929068
469,W,2026-08-02,1411.748987,208619.678055


**Q8** - Rank products by total revenue within each category (DENSE_RANK)

In [50]:
query8 = """
WITH revenue AS (
    SELECT p.category, p.product_name,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS total_revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.category, p.product_name
)
SELECT category, product_name, total_revenue,
       DENSE_RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS rank_in_category
FROM revenue
ORDER BY category, rank_in_category;
"""

pd.read_sql(query8, conn)

,category,product_name,total_revenue,rank_in_category
0,Books,Someone,8413.282420,1
1,Books,Away Comic,7206.416774,2
2,Books,Since Comic,6986.902828,3
3,Books,Follow,6957.492181,4
4,Books,Safe,6488.057894,5
...,...,...,...,...
482,Home,Across,15.858322,110
483,Home,Key,11.687112,111
484,Home,Face,9.465850,112
485,Home,Listen,-216.861593,113


**Q9** - Days between consecutive orders per customer (LAG)

In [51]:
query9 = """
WITH ordered AS (
    SELECT customer_id, order_date,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_order_date
    FROM orders
    WHERE customer_id IS NOT NULL
)
SELECT customer_id, order_date, previous_order_date,
       julianday(order_date) - julianday(previous_order_date) AS days_gap
FROM ordered
WHERE previous_order_date IS NOT NULL
ORDER BY customer_id, order_date;
"""

gap_df = pd.read_sql(query9, conn)
gap_df.head()

,customer_id,order_date,previous_order_date,days_gap
0,1.0,2026-05-12 22:16:45,2025-08-10 04:32:31,275.739051
1,7.0,2026-04-07 02:11:24,2025-10-11 20:43:55,177.227419
2,7.0,2026-06-09 12:09:34,2026-04-07 02:11:24,63.415394
3,9.0,2026-05-26 11:50:54,2025-08-16 17:58:03,282.745035
4,10.0,2025-12-27 15:25:22,2025-12-26 05:06:12,1.429977


In [52]:
# Flag customers with average gap > 30 days as "At Risk"
at_risk = gap_df.groupby("customer_id")["days_gap"].mean().reset_index()
at_risk["status"] = at_risk["days_gap"].apply(lambda x: "At Risk" if x > 30 else "Active")
at_risk.head()

,customer_id,days_gap,status
0,1.0,275.739051,At Risk
1,7.0,120.321406,At Risk
2,9.0,282.745035,At Risk
3,10.0,77.926215,At Risk
4,13.0,64.335602,At Risk


**Q10** - CTE with multiple levels - revenue category count per month

In [53]:
query10 = """
WITH monthly_revenue AS (
    SELECT o.customer_id,
           strftime('%Y-%m', o.order_date) AS month,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.customer_id IS NOT NULL
    GROUP BY o.customer_id, month
),
categorized AS (
    SELECT month, customer_id,
           CASE
               WHEN revenue > 10000 THEN 'High'
               WHEN revenue >= 5000 THEN 'Medium'
               ELSE 'Low'
           END AS category
    FROM monthly_revenue
)
SELECT month, category, COUNT(*) AS customer_count
FROM categorized
GROUP BY month, category
ORDER BY month, category;
"""

pd.read_sql(query10, conn)

,month,category,customer_count
0,2025-08,Low,28
1,2025-08,Medium,1
2,2025-09,Low,28
3,2025-09,Medium,2
4,2025-10,Low,51
5,2025-10,Medium,4
6,2025-11,Low,40
7,2025-11,Medium,2
8,2025-12,Low,31
9,2025-12,Medium,4


**Q11** - NTILE for customer segmentation (Platinum/Gold/Silver/Bronze)

In [54]:
query11 = """
WITH customer_value AS (
    SELECT o.customer_id,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS total_value
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.customer_id IS NOT NULL
    GROUP BY o.customer_id
)
SELECT customer_id, total_value,
       NTILE(4) OVER (ORDER BY total_value DESC) AS quartile
FROM customer_value;
"""

quartile_df = pd.read_sql(query11, conn)

labels = {1: "Platinum", 2: "Gold", 3: "Silver", 4: "Bronze"}
quartile_df["quartile_label"] = quartile_df["quartile"].map(labels)
quartile_df.head()

,customer_id,total_value,quartile,quartile_label
0,430.0,14298.559379,1,Platinum
1,257.0,11749.655188,1,Platinum
2,406.0,11629.061731,1,Platinum
3,61.0,11227.834712,1,Platinum
4,356.0,11108.293241,1,Platinum


**Q12** - Year-over-year revenue comparison

In [55]:
query12 = """
WITH monthly AS (
    SELECT strftime('%Y', o.order_date) AS year,
           strftime('%m', o.order_date) AS month,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY year, month
)
SELECT curr.year, curr.month, curr.revenue,
       prev.revenue AS prev_year_revenue,
       CASE WHEN prev.revenue IS NOT NULL AND prev.revenue != 0
            THEN (curr.revenue - prev.revenue) * 100.0 / prev.revenue
            ELSE NULL END AS yoy_growth_percent
FROM monthly curr
LEFT JOIN monthly prev
    ON curr.month = prev.month AND CAST(curr.year AS INTEGER) = CAST(prev.year AS INTEGER) + 1
ORDER BY curr.year, curr.month;
"""

pd.read_sql(query12, conn)

,year,month,revenue,prev_year_revenue,yoy_growth_percent
0,2025,08,51120.306491,NaN,NaN
1,2025,09,73313.240406,NaN,NaN
2,2025,10,120952.116266,NaN,NaN
3,2025,11,82404.958968,NaN,NaN
4,2025,12,75863.137197,NaN,NaN
5,2026,01,100033.159567,NaN,NaN
6,2026,02,80868.237285,NaN,NaN
7,2026,03,67773.212799,NaN,NaN
8,2026,04,74458.133764,NaN,NaN
9,2026,05,85581.388930,NaN,NaN


**Q13** - First and most recent purchased category per customer

In [56]:
query13 = """
WITH purchases AS (
    SELECT o.customer_id, o.order_date, p.category,
           ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY o.order_date ASC) AS rn_first,
           ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY o.order_date DESC) AS rn_last
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    WHERE o.customer_id IS NOT NULL
),
first_cat AS (SELECT customer_id, category AS first_category FROM purchases WHERE rn_first = 1),
last_cat AS (SELECT customer_id, category AS last_category FROM purchases WHERE rn_last = 1)
SELECT f.customer_id, f.first_category, l.last_category,
       CASE WHEN f.first_category != l.last_category THEN 'Yes' ELSE 'No' END AS category_shift
FROM first_cat f
JOIN last_cat l ON f.customer_id = l.customer_id;
"""

pd.read_sql(query13, conn)

,customer_id,first_category,last_category,category_shift
0,1.0,Electronics,Electronics,No
1,2.0,Home,Home,No
2,6.0,Clothing,Clothing,No
3,7.0,Electronics,Home,Yes
4,8.0,Electronics,Electronics,No
...,...,...,...,...
319,507.0,Electronics,Electronics,No
320,508.0,Home,Electronics,Yes
321,511.0,Electronics,Electronics,No
322,517.0,Clothing,Clothing,No


**Q14** - Cumulative revenue distribution (top N% of customers)

In [57]:
query14 = """
WITH customer_revenue AS (
    SELECT o.customer_id,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.customer_id IS NOT NULL
    GROUP BY o.customer_id
)
SELECT customer_id, revenue,
       SUM(revenue) OVER (ORDER BY revenue DESC) AS cumulative_revenue,
       SUM(revenue) OVER (ORDER BY revenue DESC) * 100.0 / SUM(revenue) OVER () AS cumulative_percent
FROM customer_revenue
ORDER BY revenue DESC;
"""

pd.read_sql(query14, conn)

,customer_id,revenue,cumulative_revenue,cumulative_percent
0,430.0,14298.559379,14298.559379,1.489219
1,257.0,11749.655188,26048.214567,2.712964
2,406.0,11629.061731,37677.276298,3.924150
3,61.0,11227.834712,48905.111010,5.093548
4,356.0,11108.293241,60013.404251,6.250495
...,...,...,...,...
319,176.0,-124.144224,964432.232272,100.447207
320,508.0,-197.460677,964234.771595,100.426641
321,174.0,-838.156683,963396.614912,100.339345
322,420.0,-1427.988927,961968.625985,100.190618


**Q15** - Cohort analysis - retention by registration month

In [58]:
query15 = """
WITH cohort AS (
    SELECT customer_id, strftime('%Y-%m', registration_date) AS cohort_month
    FROM customers
),
orders_month AS (
    SELECT o.customer_id, strftime('%Y-%m', o.order_date) AS order_month
    FROM orders o
    WHERE o.customer_id IS NOT NULL
),
joined AS (
    SELECT c.cohort_month, om.order_month,
           (CAST(strftime('%Y', om.order_month || '-01') AS INTEGER) * 12 +
            CAST(strftime('%m', om.order_month || '-01') AS INTEGER)) -
           (CAST(strftime('%Y', c.cohort_month || '-01') AS INTEGER) * 12 +
            CAST(strftime('%m', c.cohort_month || '-01') AS INTEGER)) AS month_offset,
           c.customer_id
    FROM cohort c
    JOIN orders_month om ON c.customer_id = om.customer_id
)
SELECT cohort_month, month_offset, COUNT(DISTINCT customer_id) AS customers_ordered
FROM joined
WHERE month_offset BETWEEN 0 AND 3
GROUP BY cohort_month, month_offset
ORDER BY cohort_month, month_offset;
"""

pd.read_sql(query15, conn)

,cohort_month,month_offset,customers_ordered
0,2025-05,3,2
1,2025-06,2,2
2,2025-06,3,1
3,2025-07,1,2
4,2025-07,2,1
5,2025-07,3,2
6,2025-08,0,1
7,2025-08,1,2
8,2025-08,2,2
9,2025-08,3,2


**Q16** - Products frequently bought together (self-join)

In [59]:
query16 = """
SELECT a.product_id AS product_a, b.product_id AS product_b,
       COUNT(*) AS times_bought_together
FROM order_items a
JOIN order_items b
    ON a.order_id = b.order_id
    AND a.product_id < b.product_id
GROUP BY a.product_id, b.product_id
ORDER BY times_bought_together DESC
LIMIT 20;
"""

pd.read_sql(query16, conn)

,product_a,product_b,times_bought_together
0,9,135,2
1,41,218,2
2,80,179,2
3,80,437,2
4,99,446,2
5,123,167,2
6,132,316,2
7,176,498,2
8,177,488,2
9,182,446,2


# Part 4: Python + SQL Integration (CLI-style tool)

Written as a function so it can be called directly inside the notebook.
Run it as a real command-line script separately using `input()` if required.

In [60]:
def generate_report(report_type, start_date, end_date, conn):
    query = """
    SELECT o.order_id, o.order_date, oi.quantity, oi.unit_price, oi.discount_percent, o.customer_id, oi.product_id
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE date(o.order_date) BETWEEN date(?) AND date(?)
    """
    df = pd.read_sql(query, conn, params=(start_date, end_date))

    df["revenue"] = df["quantity"] * df["unit_price"] * (1 - df["discount_percent"]/100)

    total_orders = df["order_id"].nunique()
    total_revenue = df["revenue"].sum()
    unique_customers = df["customer_id"].nunique()

    top_products = (
        df.groupby("product_id")["revenue"].sum()
        .sort_values(ascending=False)
        .head(3)
    )

    print(f"Report type: {report_type}")
    print(f"Date range: {start_date} to {end_date}")
    print(f"Total orders: {total_orders}")
    print(f"Total revenue: {total_revenue:.2f}")
    print(f"Unique customers: {unique_customers}")
    print("Top 3 products:")
    print(top_products)

    return df

# example run
report_df = generate_report("monthly", "2025-01-01", "2025-12-31", conn)

Report type: monthly
Date range: 2025-01-01 to 2025-12-31
Total orders: 214
Total revenue: 403653.76
Unique customers: 164
Top 3 products:
product_id
266    6518.745129
465    5835.380270
239    5445.080265
Name: revenue, dtype: float64


# Part 5: Edge Case Handling

In [61]:
def test_orphan_order_items():
    orphans = check_referential_integrity(orders_df, order_items_df)
    print("Orphan order_items found:", len(orphans))
    return orphans

def test_discount_over_100():
    bad = order_items_df[order_items_df["discount_percent"] > 100]
    print("Rows with discount_percent > 100:", len(bad))
    return bad

def test_zero_quantity():
    zero_qty = order_items_df[order_items_df["quantity"] == 0]
    print("Rows with quantity = 0:", len(zero_qty))
    return zero_qty

def test_future_order_date():
    future = orders_clean[orders_clean["order_date"] > datetime.now()]
    print("Rows with future order_date:", len(future))
    return future

test_orphan_order_items()
test_discount_over_100()
test_zero_quantity()
test_future_order_date()

Orphan order_items found: 0
Rows with discount_percent > 100: 0
Rows with quantity = 0: 0
Rows with future order_date: 0


,order_id,customer_id,order_date,status,region_code,missing_customer


In [62]:
conn.close()